# UC CosMx WTX: NicheNet ligand-regulator pairs in OGN⁺RSPO3⁺ fibroblasts

Builds the table of NicheNet-predicted regulators (with their upstream SELENOP⁺MMP9⁺SPP1⁺ macrophage ligands) that `11_ligand_regulator_cooccurrence.py` tests in tissue.

1. Load the annotated UC WTX object (`01_wtx_processing.ipynb`, labels from `05_aucell_annotation.R`) and keep the two inflamed UC patients.
2. Load the NicheNet results of `uc_scrna/14_nichenet_macrophage_to_fibroblast.R` (ligand activities, ligand-regulator predictions, predicted regulators, regulator-target network; saved with the `ogn_spp1_mac_` prefix).
3. Split OGN⁺RSPO3⁺ fibroblasts by distance to the nearest Mac S+M+S+ (near < 250 px) and compare each predicted regulator between near and far fibroblasts, by expression and by the score of its NicheNet targets.
4. Merge both tables into `ogn_spp1_mac_nichenet_tf_spatial_validation_combined_v2.csv` (regulator, upstream ligands, expression and activity statistics).

**Output:** `ogn_spp1_mac_nichenet_tf_spatial_validation_expression.csv`, `..._activity.csv`, `..._combined_v2.csv` (input of `11`).

In [330]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from scipy.spatial import cKDTree

In [2]:
adata_wtx=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/UC_1k_6k_wholetrans/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_anno.h5ad')

/data/ydn4687/geneformer_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [139]:
adata_wtx=adata_wtx[adata_wtx.obs['patient'].isin(['I0261','I0331'])]

In [140]:
adata_wtx

View of AnnData object with n_obs × n_vars = 61058 × 18935
    obs: 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68', 'Max.CD68', 'Mean.CD298_B2M', 'Max.CD298_B2M', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'cell_id', 'version', 'Run_name', 'Slide_name', 'Run_Tissue_name', 'Run_Slide_name', 'ISH.concentration', 'CosMx', 'tissue', 'Panel', 'assay_type', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quantile_0.9', 'negprobes_quantile_0.95', 'negprobes_quantile_0.99', 'nCount_negprobes', 'nFeature_negprobes', 'median_falsecode', 'falsecode_quantile_0.75', 'falsecode_quantile_0.8', 'falsecode_quantile_0.85', 'falsecode_quantile_0.9', 'falsecode

In [550]:
# Load NicheNet results
ligand_tf_predictions = pd.read_csv("/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_ligand_tf_predictions.csv")
predicted_tfs = pd.read_csv("/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_predicted_tfs.csv")
tf_target_network = pd.read_csv("/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_tf_target_network.csv")
ligand_activities = pd.read_csv("/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_ligand_activities.csv")

In [551]:
ligand_tf_predictions

,ligand,tf,score
0,COL3A1,MAG,0.002683
1,COL3A1,BMP1,0.002111
2,COL3A1,MMP13,0.001954
3,COL3A1,FGF7,0.001414
4,COL3A1,TNC,0.001371
...,...,...,...
1174,SDC2,PGF,0.000375
1175,SDC2,WNT7A,0.000362
1176,SDC2,HSPG2,0.000346
1177,SDC2,COL2A1,0.000339


In [552]:
predicted_tfs

,tf
0,BMP1
1,FGF7
2,TNC
3,COL4A1
4,NTN4
...,...
119,ARF1
120,ANXA1
121,FBRS
122,TNFSF13B


In [553]:
ligand_activities

,test_ligand,auroc,aupr,aupr_corrected,pearson,rank
0,COL3A1,0.587119,0.512240,0.088500,0.132183,1
1,TGFB1,0.591420,0.516606,0.092866,0.131519,2
2,GAS6,0.577461,0.497643,0.073903,0.122864,3
3,BMP1,0.575069,0.500714,0.076974,0.121546,4
4,CALM3,0.576077,0.508229,0.084489,0.119024,5
...,...,...,...,...,...,...
214,PRNP,0.555612,0.472761,0.049022,0.016171,215
215,PLAU,0.534083,0.454543,0.030803,0.008244,216
216,ST6GAL1,0.560473,0.475713,0.051973,-0.000873,217
217,CD274,0.559140,0.464284,0.040544,-0.019498,218


In [554]:
tf_target_network

,from,to,source,database
0,ADAM17,IL6,lr_evex_regulation_expression,evex_expression
1,ADAM17,TNF,lr_evex_regulation_expression,evex_expression
2,ADAM17,ALOX5AP,lr_evex_regulation_expression,evex_expression
3,ADAM17,ALPPL2,lr_evex_regulation_expression,evex_expression
4,ADAM17,APP,lr_evex_regulation_expression,evex_expression
...,...,...,...,...
1466,WNT5A,HOXA11,lr_pathwaycommons_controls_expression_of,pathwaycommons_expression
1467,WNT5A,KRT1,lr_pathwaycommons_controls_expression_of,pathwaycommons_expression
1468,WNT5A,LOR,lr_pathwaycommons_controls_expression_of,pathwaycommons_expression
1469,WNT5A,SOD2,lr_pathwaycommons_controls_expression_of,pathwaycommons_expression


In [555]:


print(f"NicheNet predicted {len(predicted_tfs)} TFs")
print("\nTop predicted TFs:")
print(predicted_tfs.head(20))

print("\nTop ligands by activity:")
print(ligand_activities.head(10))

NicheNet predicted 124 TFs

Top predicted TFs:
         tf
0      BMP1
1      FGF7
2       TNC
3    COL4A1
4      NTN4
5     LAMB2
6     LAMA2
7   COL18A1
8     LAMC3
9    COL1A1
10    HSPG2
11   COL5A3
12    NAMPT
13    TIMP1
14    LAMB1
15    THBS1
16    SFRP2
17    ITGB1
18    TGFB1
19     GAS6

Top ligands by activity:
  test_ligand     auroc      aupr  aupr_corrected   pearson  rank
0      COL3A1  0.587119  0.512240        0.088500  0.132183     1
1       TGFB1  0.591420  0.516606        0.092866  0.131519     2
2        GAS6  0.577461  0.497643        0.073903  0.122864     3
3        BMP1  0.575069  0.500714        0.076974  0.121546     4
4       CALM3  0.576077  0.508229        0.084489  0.119024     5
5        MMP1  0.573386  0.503681        0.079941  0.117248     6
6         CFH  0.568918  0.500691        0.076951  0.116309     7
7      S100A4  0.574625  0.498375        0.074635  0.114724     8
8        PSAP  0.579803  0.504001        0.080261  0.111440     9
9      SEMA3C  

In [556]:
# Cell masks for proximity
ogn_fib_mask = adata_wtx.obs['aucell_cell_type_short'] == 'OGN+RSPO3+ Fib'
spp1_mac_mask = adata_wtx.obs['aucell_cell_type_short'] == 'Mac S+M+S+'


In [557]:
all_ogn_coords = adata_wtx[ogn_fib_mask].obsm['spatial_fov']
all_spp1_coords = adata_wtx[spp1_mac_mask].obsm['spatial_fov']
# Get coordinates

In [558]:
tree = cKDTree(all_spp1_coords)
distances, _ = tree.query(all_ogn_coords, k=1)

In [559]:
print(f"\nDistance statistics:")
print(f"  Min: {distances.min():.2f}")
print(f"  Max: {distances.max():.2f}")
print(f"  Mean: {distances.mean():.2f}")
print(f"  Median: {np.median(distances):.2f}")



Distance statistics:
  Min: 48.33
  Max: 2704.17
  Mean: 594.91
  Median: 530.54


In [560]:
# Create OGN+ fibroblast subset with distance info
adata_ogn = adata_wtx[ogn_fib_mask].copy()
adata_ogn.obs['distance_to_spp1_mac'] = distances

In [561]:
threshold = 250  # pixels (spatial_fov coordinates)
adata_ogn.obs['proximity_to_spp1_mac'] = 'far'
adata_ogn.obs.loc[adata_ogn.obs['distance_to_spp1_mac'] < threshold, 'proximity_to_spp1_mac'] = 'near'

In [562]:
print(f"\nOGN+ fibroblasts near SPP1+ macs: {(adata_ogn.obs['proximity_to_spp1_mac'] == 'near').sum()}")
print(f"OGN+ fibroblasts far from SPP1+ macs: {(adata_ogn.obs['proximity_to_spp1_mac'] == 'far').sum()}")


OGN+ fibroblasts near SPP1+ macs: 120
OGN+ fibroblasts far from SPP1+ macs: 522


In [563]:
# Test TF expression near vs far
predicted_tf_list = predicted_tfs['tf'].tolist()

In [564]:
tf_expression_results = []

for tf in predicted_tf_list:
    if tf in adata_ogn.var_names:
        near_expr = adata_ogn[adata_ogn.obs['proximity_to_spp1_mac'] == 'near', tf].X
        far_expr = adata_ogn[adata_ogn.obs['proximity_to_spp1_mac'] == 'far', tf].X
        
        if hasattr(near_expr, 'toarray'):
            near_expr = near_expr.toarray().flatten()
            far_expr = far_expr.toarray().flatten()
        
        stat_val, pval = stats.ttest_ind(near_expr, far_expr)
        
        # Get upstream ligands from NicheNet
        upstream = ligand_tf_predictions[ligand_tf_predictions['tf'] == tf]['ligand'].unique()
        
        tf_expression_results.append({
            'TF': tf,
            'mean_near': near_expr.mean(),
            'mean_far': far_expr.mean(),
            'log2FC': np.log2((near_expr.mean() + 1) / (far_expr.mean() + 1)),
            'pvalue': pval,
            'pct_near': (near_expr > 0).sum() / len(near_expr) * 100,
            'pct_far': (far_expr > 0).sum() / len(far_expr) * 100,
            'upstream_ligands': ','.join(upstream[:3]) if len(upstream) > 0 else 'none'
        })


In [565]:
tf_expr_df = pd.DataFrame(tf_expression_results)
tf_expr_df['padj'] = stats.false_discovery_control(tf_expr_df['pvalue'])
tf_expr_df = tf_expr_df.sort_values('log2FC', ascending=False)

In [566]:
print("\n=== TF Expression: Near vs Far (from NicheNet predictions) ===")
print(tf_expr_df.head(30))
print(f"\nSignificant TFs (padj < 0.05): {(tf_expr_df['padj'] < 0.05).sum()}")
print(tf_expr_df[tf_expr_df['padj'] < 0.05])



=== TF Expression: Near vs Far (from NicheNet predictions) ===
            TF  mean_near  mean_far    log2FC    pvalue   pct_near    pct_far  \
118      ANXA1   0.120318  0.069926  0.066397  0.054762  12.500000   9.770115   
6        LAMA2   0.084545  0.043503  0.055656  0.048203   9.166667   6.896552   
84       GSTP1   0.153471  0.111634  0.053300  0.176001  16.666667  14.942529   
63       DCHS1   0.089616  0.051000  0.052058  0.086532  10.833333   6.704981   
78       FGF13   0.073455  0.037343  0.049368  0.049072  10.000000   5.555556   
45       THBS2   0.147449  0.112475  0.044657  0.329040  14.166667  11.494253   
29        HAS2   0.084383  0.053876  0.041169  0.158064  11.666667   7.854406   
2          TNC   0.100014  0.070469  0.039279  0.239878  12.500000   9.195402   
60       NUCB2   0.125011  0.096398  0.037167  0.337001  12.500000  12.643678   
11      COL5A3   0.066224  0.041615  0.033688  0.209853   8.333333   5.747126   
96       CCL19   0.219701  0.193587  0.031224

In [487]:
tf_expr_df.to_csv('/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_tf_spatial_validation_expression.csv', index=False)


In [567]:
# Test TF ACTIVITY using target genes
tf_activity_results = []

for tf in predicted_tf_list:
    # Get target genes for this TF from NicheNet
    targets = tf_target_network[tf_target_network['from'] == tf]['to'].tolist()
    targets_present = [t for t in targets if t in adata_ogn.var_names]
    
    if len(targets_present) < 3:
        continue
    
    # Score TF activity
    sc.tl.score_genes(
        adata_ogn,
        gene_list=targets_present,
        score_name=f'{tf}_activity',
        use_raw=False
    )
    
    near_activity = adata_ogn[adata_ogn.obs['proximity_to_spp1_mac'] == 'near'].obs[f'{tf}_activity']
    far_activity = adata_ogn[adata_ogn.obs['proximity_to_spp1_mac'] == 'far'].obs[f'{tf}_activity']
    
    stat_val, pval = stats.ttest_ind(near_activity, far_activity)
    
    upstream = ligand_tf_predictions[ligand_tf_predictions['tf'] == tf]['ligand'].unique()
    
    tf_activity_results.append({
        'TF': tf,
        'n_targets': len(targets_present),
        'mean_activity_near': near_activity.mean(),
        'mean_activity_far': far_activity.mean(),
        'activity_diff': near_activity.mean() - far_activity.mean(),
        'pvalue': pval,
        'upstream_ligands': ','.join(upstream[:3]) if len(upstream) > 0 else 'none'
    })


In [568]:
tf_activity_df = pd.DataFrame(tf_activity_results)
tf_activity_df['padj'] = stats.false_discovery_control(tf_activity_df['pvalue'])
tf_activity_df = tf_activity_df.sort_values('activity_diff', ascending=False)

print("\n=== TF Activity: Near vs Far (from NicheNet target genes) ===")
print(tf_activity_df.head(20))
print(f"\nSignificant TFs by activity (padj < 0.05): {(tf_activity_df['padj'] < 0.05).sum()}")
print(tf_activity_df[tf_activity_df['padj'] < 0.05])



=== TF Activity: Near vs Far (from NicheNet target genes) ===
         TF  n_targets  mean_activity_near  mean_activity_far  activity_diff  \
43   CXCL13          3            0.014487          -0.003385       0.017871   
3     LAMC3          4            0.014633          -0.003022       0.017655   
21    PSEN1          8            0.011979          -0.002788       0.014767   
42    CCL11          5            0.008955          -0.002469       0.011424   
57  HLA-DRA          3            0.010654          -0.000374       0.011027   
52    CCL19          3            0.006207          -0.001474       0.007681   
44   ADAM17         48            0.002229          -0.003848       0.006077   
55   LRPAP1          3            0.004156          -0.000250       0.004406   
27     TSLP         25           -0.004164          -0.008285       0.004120   
20    HMGB1         50            0.031816           0.027808       0.004008   
51      MIF         46           -0.000432          -0.00

In [569]:
tf_activity_df

,TF,n_targets,mean_activity_near,mean_activity_far,activity_diff,pvalue,upstream_ligands,padj
43,CXCL13,3,0.014487,-0.003385,0.017871,0.174469,CALM1,0.745413
3,LAMC3,4,0.014633,-0.003022,0.017655,0.092058,"COL3A1,COL1A2,LAMB3",0.539856
21,PSEN1,8,0.011979,-0.002788,0.014767,0.099666,"S100A4,C1QB,APOC2",0.539856
42,CCL11,5,0.008955,-0.002469,0.011424,0.317334,"CALM1,MMP2",0.791278
57,HLA-DRA,3,0.010654,-0.000374,0.011027,0.470365,"CD1B,HLA-DOA",0.852175
...,...,...,...,...,...,...,...,...
28,PSAP,4,-0.024174,-0.005679,-0.018494,0.211283,PSAP,0.766440
17,RARRES2,5,-0.029410,-0.007029,-0.022381,0.056219,"MMP1,S100A8,CTSD",0.456776
40,CALM1,5,-0.017816,0.005895,-0.023712,0.049757,"LGALS3BP,CALM1,CD209",0.456776
11,CALM3,5,-0.019873,0.006177,-0.026050,0.026879,"CALM3,S100A4,CALM1",0.353349


In [490]:
tf_activity_df.to_csv('/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_tf_spatial_validation_activity.csv', index=False)

In [570]:
# Combine results
combined = tf_expr_df[['TF', 'log2FC', 'padj', 'pvalue','upstream_ligands']].merge(
    tf_activity_df[['TF', 'activity_diff', 'padj','pvalue']],
    on='TF',
    suffixes=('_expr', '_activity')
)

combined

,TF,log2FC,padj_expr,pvalue_expr,upstream_ligands,activity_diff,padj_activity,pvalue_activity
0,ANXA1,0.066397,0.608976,0.054762,ITGA4,0.000935,0.995600,0.915866
1,LAMA2,0.055656,0.608976,0.048203,"COL3A1,COL1A2,LIN7C",0.000020,0.995600,0.995600
2,TNC,0.039279,0.701717,0.239878,"COL3A1,MMP1,COL1A2",-0.017848,0.353349,0.025066
3,CCL19,0.031224,0.889035,0.550585,MMP2,0.007681,0.868480,0.547811
4,TNFSF12,0.028385,0.762231,0.357633,"MMP1,MMP14",-0.002978,0.852175,0.495666
...,...,...,...,...,...,...,...,...
59,TGFB3,-0.060106,0.692535,0.087370,"MMP14,MMP25,ENG",-0.004867,0.791278,0.337905
60,VCAM1,-0.064593,0.608976,0.052759,"ITGA9,ITGA4",-0.015409,0.791278,0.271613
61,CALM2,-0.088565,0.608976,0.043603,"CALM1,CTSD",-0.038197,0.353349,0.012634
62,SFRP2,-0.093361,0.692535,0.111484,COL3A1,0.000911,0.995600,0.945635


In [587]:

combined.to_csv('/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_tf_spatial_validation_combined_v2.csv', index=False)